# LangSmith: a hands-on implementation

**Track:** Frameworks · **Guide:** [`056-langsmith.md`](../guides/concepts/056-langsmith.md)

LangSmith supports tracing, dataset-based evaluation, and monitoring of LLM application runs.

> **Intuition:** A flight recorder and test lab let an engineer inspect each component of a model workflow.

This lesson builds a small working example in stages: prepare inputs, implement
the core idea, run it on a realistic toy case, and inspect its output and limits.
It runs offline unless a cell is explicitly marked as an optional provider call.

```mermaid
flowchart LR
    A[Prepare inputs] --> B[Implement LangSmith]
    B --> C[Run on example]
    C --> D[Evaluate and inspect]
```

**Case study:** Trace retrieval and generation separately to find whether a wrong answer came from search, context, or the model.

Run the cells from top to bottom. To install the notebook environment, follow
[`guides/SETUP.md`](../guides/SETUP.md). The toy data and implementation are for
learning, not a production quality, safety, or security guarantee.


## 1. Represent a run as a trace

Tracing records inputs, outputs, timing, and errors for each component in an LLM application.


In [ ]:
from dataclasses import dataclass, field
from time import perf_counter


@dataclass
class Trace:
    name: str
    inputs: dict
    outputs: dict = field(default_factory=dict)
    elapsed_ms: float = 0.0

def traced(name, function, **inputs):
    started = perf_counter()
    output = function(**inputs)
    return Trace(name, inputs, {"result": output}, (perf_counter() - started) * 1000)

## 2. Trace retrieval and answer stages

Separate spans identify whether a bad response started with retrieval or generation. This recorder is local; it does not send telemetry.


In [ ]:
def search(query):
    return ["Returns are accepted within 30 days."] if "return" in query.lower() else []

def answer(context):
    return context[0] if context else "I don't know."

retrieval_trace = traced("retrieve", search, query="What is the return period?")
answer_trace = traced(
    "answer", answer, context=retrieval_trace.outputs["result"]
)
trace_tree = [retrieval_trace, answer_trace]

## 3. Evaluate and inspect traces

Dataset-based evaluation compares predictions with expected behavior; production monitoring should also capture latency and failure rates.


In [ ]:
expected = "Returns are accepted within 30 days."
actual = answer_trace.outputs["result"]
evaluation = {"exact_match": actual == expected, "latency_ms": sum(t.elapsed_ms for t in trace_tree)}
for trace in trace_tree:
    print(trace.name, round(trace.elapsed_ms, 3), "ms", trace.outputs)
print("evaluation:", evaluation)

## 4. Optionally send traces to LangSmith

Install the `langsmith` extra and configure LangSmith tracing before opting in. Never send sensitive prompts or data without approval.


In [ ]:
RUN_LANGSMITH = False
if RUN_LANGSMITH:
    from examples.integrations.langsmith_example import answer_from_evidence

    print(answer_from_evidence("return window", [expected]))
else:
    print("LangSmith tracing skipped; the trace above stayed local.")

## Wrap-up and next steps

- Identify what was learned, stored, retrieved, or generated in this implementation.
- Change an input and observe how the output changes.
- Test one failure case and decide when the system should abstain or ask for help.
- Replace toy data/components only after you can evaluate the change against a
  held-out or representative test set.

Continue with the [concept guide](../guides/concepts/056-langsmith.md) and the
[learning roadmap](../guides/ROADMAP.md).
